# OpenPlaque — LAD Takeoff Focused Confirmation v2

Freeze the strongest prior root-directed alternative (alternative 1). Test only (1) candidate-to-root trunk continuity and (2) persistence of one independent non-LAD coronary-like branch for 6–10 mm. v2 fixes the initial branch-separation geometry while retaining the final independence gate. No global LAD search and no automatic LCX label. Research use only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls — True + valid cache reuses; False forces recomputation/overwrite.
REUSE_SOURCE_CT = True
REUSE_PRIOR_CANDIDATE = True
REUSE_AORTA_CONSTRAINT = True
REUSE_TRUNK_QC = True
REUSE_SECONDARY_BRANCH = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install dependencies


In [ ]:
%pip -q install scipy matplotlib pandas
print('Dependencies ready.')


## Step 4 — Load this fresh branch


In [ ]:
import os, sys, subprocess, shutil
REPO='/content/OpenPlaque'
BRANCH='lad-takeoff-confirmation-from-main'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',REPO],check=True)
sys.path.insert(0,os.path.join(REPO,'src'))
print('Loaded',BRANCH)


## Step 5 — Initialize and inspect caches


In [ ]:
from openplaque.lad_takeoff_confirmation_v2 import LADTakeoffConfirmationWorkflow
reuse={
 'source_ct':REUSE_SOURCE_CT,
 'prior_candidate':REUSE_PRIOR_CANDIDATE,
 'aorta_constraint':REUSE_AORTA_CONSTRAINT,
 'trunk_qc':REUSE_TRUNK_QC,
 'secondary_branch':REUSE_SECONDARY_BRANCH,
 'figures':REUSE_FIGURES,
 'report':REUSE_REPORT,
}
wf=LADTakeoffConfirmationWorkflow(reuse=reuse)
display(wf.cache_status())


## Step 6 — Load source CCTA and freeze prior candidate

This reuses the disk-backed series-7 CCTA, alternative 1, the candidate takeoff location, the validated RCA calibration, and the existing TotalSegmentator aorta distance/exclusion constraint. It does not rediscover the LAD.


In [ ]:
wf.load_source_ct()
prior=wf.load_prior_candidate()
wf.load_aorta_constraint()
print('Prior takeoff candidate:',prior)
from openplaque.lad_takeoff_confirmation import arc_mm
print('Trunk length to candidate:',float(arc_mm(wf.trunk,wf.spacing)[-1]),'mm')


## Step 7 — Test candidate-to-root trunk continuity


In [ ]:
trunk=wf.evaluate_trunk()
print(trunk)
if not trunk['accepted']:
    print('TRUNK GATE DID NOT PASS — secondary branch will still be measured, but final status cannot be supported.')


## Step 8 — Trace one independent secondary branch

Search starts exactly at the takeoff candidate. The first step is allowed to remain near the bifurcation, then the independence requirement increases with distance from the origin. Final PASS still requires a persistent ≥6 mm coronary-like branch and ≥2.3 mm endpoint separation from the known LAD/trunk. The branch is not labeled LCX.


In [ ]:
branch=wf.trace_secondary_branch(max_length_mm=10.0,beam_width=12)
print(branch)
if wf.branch_candidates is not None and len(wf.branch_candidates):
    display(wf.branch_candidates.head(12))


## Step 9 — Focused confirmation decision


In [ ]:
summary=wf.summarize()
print(summary)


## Step 10 — Generate QC figures


In [ ]:
from IPython.display import Image, display
figs=wf.plot_qc()
for f in figs:
    print(f)
    display(Image(filename=str(f)))


## Step 11 — Package report-back ZIP


In [ ]:
z=wf.package()
print('REPORT BACK:',z)
print('Expected file: OPENPLAQUE_LAD_TAKEOFF_CONFIRMATION_REPORT_BACK.zip')
print('After this finishes, tell ChatGPT: Retrieve and analyze')
